# Welcome to the IMC Trading Challenge!


# Some Notes

## Exchange GUI

- The test exchange is at https://testcmiexchange.imclaunchpad.com
- The real exchange is at https://cmiexchange.imclaunchpad.com

## Optional features

Sections marked as optional below are features supported by the exchange but not necessarily needed for writing a working arbitrager script. However, you are more than welcome to try it!

## Rules

- The aim of this challenge is to write an automated algorithm to trade
- Once the competition is completed, we will inspect the code to ensure that you've acted in good faith. We reserve the right to disqualify if we determine this is not the case.
- We do not impose a hard limit on position, but try to keep it flat.


## Setting up

We first need an Exchangebot to send orders and read market data. Please note that you will be default connect to the test exchange.


In [ ]:
%pip install requests
%pip install sseclient-py

In [ ]:
import sys
from pathlib import Path

_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "base_bot.py").exists())
sys.path[:0] = [str(_root), str(Path.cwd())]

In [ ]:
from student_bot import StudentBot
from models import Order, OrderBook, OrderRequest, Side


In [ ]:
username = "TODO"  # TODO: Change this to be the username you've created in CMI
password = "TODO"  # TODO: Change this to be the password you've created in CMI
exchange_url = "https://testcmiexchange.imclaunchpad.com"

In [ ]:
class Bot(StudentBot):
    def on_orderbooks(self, orderbooks: dict[str, OrderBook]):
        pass


bot = Bot(exchange_url, username, password)

## Viewing the market

We can view the current market state on every orderbook change in the `on_orderbooks` handler.


In [ ]:
%pdef bot.on_orderbook
%pdoc OrderBook

From the output above, we can see that the on_orderbook handler will give you an orderbook with `buy_orders` sorted by decreasing price and `sell_orders` sorted by increasing price.

Here is an example of the orderbooks `on_orderbooks` gives you and how to access the fields of an orderbook below.


In [ ]:
orderbooks: dict[str, OrderBook] = {
    "PRODUCT": OrderBook(
        product="PRODUCT",
        tick_size=1,
        buy_orders=[
            Order(price=100, volume=1, own_volume=1),
            Order(price=99, volume=1, own_volume=0),
        ],
        sell_orders=[
            Order(price=101, volume=1, own_volume=1),
            Order(price=102, volume=1, own_volume=0),
        ],
    )
}
orderbook = orderbooks["PRODUCT"]
orderbook.buy_orders[0]

The first quote in the buy orders gives the current best `bid` price, `orderbook.buy_orders[0].price`, and the quantity someone is willing to buy at.

And the first quote in the sell orders gives the current best `ask` price and volume someone is willing to sell at.


In [ ]:
orderbook.buy_orders[0].price, orderbook.buy_orders[0].volume

## Sending Orders

Now that we've set up our bot, let's get into making some orders!

### Hit - Immediate or Cancel Orders (IOC)

IOC orders are orders that will either be filled (either partially or fully), and if not, cancelled. The remaining units that cannot be immediately filled will not remain on the market. This is commonly known as "hitting".


In [ ]:
%pdef bot.hit
# Send an IOC order to Sell 1 lettuce at price 500
bot.hit(OrderRequest(product="LETTUCE", side=Side.SELL, price=500, volume=1))

### Quote - Good For Day Orders (GFD) (Optional)

GFD orders are orders that will remain on the market until they are fully filled. This is commonly known as 'quoting', and we can make quotes with our instance `bot` that we defined above.


In [ ]:
%pdef bot.quote
# Send a GFD order to Buy 10 lettuce at price 5
gfd_order = bot.quote(
    OrderRequest(product="LETTUCE", price=5, side=Side.BUY, volume=10)
)
gfd_order

Note here in the return value of both `bot.quote` and `bot.hit`, there is a `filled` attribute that tells you the volume filled when you sent the order.


## Cancel an Order (Optional)

When we make an order, we get an `orderResponse` dataclass object. We can use this `orderResponse` to cancel any orders we think might be no longer attractive to keep on the market. Note that we can only cancel GFD orders, as IOC orders will be automatically cancelled if they aren't fully filled.


In [ ]:
%pdef bot.cancel_order_by_id
bot.cancel_order_by_id(gfd_order.id)

### Mass Cancel All Orders (Optional)

If the market has shifted too quickly for us, we can cancel all the quotes we have on the market. We can cancel for every symbol or for a particular symbol and price.


In [ ]:
%pdef bot.cancel_all_orders
bot.cancel_all_orders()

In [ ]:
%pdef bot.cancel_order
bot.cancel_order(product="LETTUCE", price=101)

## View your current orders (Optional)

We can also view the current orders we've made on the market.


In [ ]:
bot.request_all_orders()

### Mass Quote (Optional)

Similiar to a mass cancel, we can also mass quote. We define a list of all the GFD orders we want to send as a list and call `bot.send_mass_orders()`


In [ ]:
%pdef bot.send_mass_orders
bot.send_mass_orders(
    [
        OrderRequest(
            product="LETTUCE", price=500, side=Side.BUY, volume=100
        ),  # GFD order to buy 100 Lettuce at 500
        OrderRequest(
            product="CHICKEN", price=500, side=Side.BUY, volume=100
        ),  # GFD order to buy 100 Chicken at 500
        OrderRequest(
            product="SALAD", price=1100, side=Side.SELL, volume=100
        ),  # GFD order to sell 1100 Salad at 1100
    ]
)